# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR² colorectal cancer survivors dataset using the `mlcroissant` library. All entities—record sets, fields, columns—are referenced by their `@id` fields for clarity and reproducibility.

### Dataset Source
The dataset and its Croissant schema are accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is available
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display basic dataset information
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"\nIdentifier: {getattr(meta, 'identifier', None)}")
print(f"Version: {getattr(meta, 'version', None)}")
print(f"License: {getattr(meta, 'license', None)}")

## 2. Data Overview
We'll discover the available record sets, field `@id`s, and their descriptions in this package.

**Note:** All schema elements are always referenced by their `@id`.

In [ ]:
# List record set @ids and their fields
record_sets = list(dataset.list_record_sets())
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    rs_info = dataset.get_record_set(rs)
    print(f"- Record set @id: {rs}\n  Name: {getattr(rs_info, 'name', None)}")
    print("  Fields:")
    for field in rs_info.fields:
        print(f"    - Field @id: {field['@id']}")
          # Field name and type if available
        fn = field.get('name', '(no name)')
        ftype = field.get('dataType', '(no type)')
        print(f"      Name: {fn}; Type: {ftype}")
    print()

## 3. Data Extraction
Load each record set into a pandas DataFrame for further analysis.
Assign variables for the `@id` of the record set and fields of interest (chosen from step 2).

Let's extract all available record sets and show the columns in one of them.

In [ ]:
# Extract all available record sets into DataFrames by @id
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs))
    df = pd.DataFrame(records)
    dataframes[rs] = df
    print(f"Loaded record set {rs}: {df.shape[0]} records, {df.shape[1]} columns.")

# Pick the first record set for demonstration (customize as needed)
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id is not None:
    print("\nColumns in main DataFrame:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate common analytical steps with this biomedical dataset: filtering, normalizing numeric columns, and grouping.

> **Note:** We **use only `@id`** for each variable. Please replace `<numeric_field_id>` and `<group_field_id>` accordingly based on the DataFrame columns printed above.

In [ ]:
# Example: Suppose there's a numeric field @id representing 'age' and a group field @id for 'Sex'
# You can update these variable values after inspecting DataFrame columns above.

record_set_id = main_record_set_id  # use the main record set

# SET THE FOLLOWING @ids BASED ON THE OUTPUT ABOVE
numeric_field_id = '<replace_with_age_field_@id>'  # e.g. 'age' field @id
group_field_id = '<replace_with_sex_field_@id>'    # e.g. 'sex' field @id

df = dataframes[record_set_id]

if numeric_field_id in df.columns:
    threshold = 50  # Example: filter age > 50
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    # Normalize numeric field
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())/filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by group field
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df)
else:
    print(f"Field {numeric_field_id} not found in record set DataFrame.")

## 5. Visualization
Let's visualize the distribution of the chosen numeric variable, and show group differences.
> Update `numeric_field_id` and `group_field_id` below as before.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if the necessary fields exist
if numeric_field_id in df.columns:
    plt.figure(figsize=(10, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    if group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print(f"Cannot plot distributions: field(s) missing from DataFrame.")

## 6. Conclusion

In this notebook, we've:
- Loaded the FAIR² colorectal cancer survivors clinical dataset via its Croissant schema
- Explored the dataset structure and referenced all entities by their `@id`
- Extracted and loaded record sets into DataFrames for analysis
- Demonstrated EDA and normalization using `@id` fields only
- Visualized the distribution and group differences of a numeric variable

For in-depth analysis or ML modeling, continue referencing fields and subsets using their Croissant-assigned `@id`s. For guidance, see the [Croissant documentation](https://mlcommons.org/croissant/).
